# Retail Data Quality — Data Cleaning

## Objective

Clean the raw retail sales dataset while preserving data lineage and documenting every major cleaning decision.

The cleaning process will:

- Remove exact duplicate records
- Identify conflicting duplicate transaction IDs
- Standardize data types
- Standardize categorical values
- Handle missing values appropriately
- Validate numeric ranges
- Handle invalid dates
- Resolve or quarantine referential-integrity issues
- Recalculate derived financial fields
- Create a review dataset for records that cannot be safely corrected
- Produce a cleaning audit report

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

In [ ]:
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"

CLEANED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
sales = pd.read_csv(
    RAW_DIR / "fact_sales.csv",
    low_memory=False
)

products = pd.read_csv(
    RAW_DIR / "dim_product.csv"
)

customers = pd.read_csv(
    RAW_DIR / "dim_customer.csv"
)

stores = pd.read_csv(
    RAW_DIR / "dim_store.csv"
)

print("Sales:", sales.shape)
print("Products:", products.shape)
print("Customers:", customers.shape)
print("Stores:", stores.shape)

### Create a row-level lineage ID

In [ ]:
sales["source_row_id"] = np.arange(
    1,
    len(sales) + 1
)

In [ ]:
cleaned_sales = sales.copy()

print(
    "Starting rows:",
    len(cleaned_sales)
)

In [ ]:
cleaning_audit = []

## Cleaning decision 
### 1 - Exact duplicates

In [ ]:
exact_duplicate_mask = cleaned_sales.duplicated(
    keep="first"
)

exact_duplicate_count = exact_duplicate_mask.sum()

print(
    "Exact duplicate rows:",
    exact_duplicate_count
)

In [ ]:
cleaned_sales = cleaned_sales[
    ~exact_duplicate_mask
].copy()

In [ ]:
cleaning_audit.append({
    "issue": "Exact duplicate rows",
    "action": "Removed duplicate copies",
    "records_affected": int(exact_duplicate_count),
    "business_reason": (
        "Exact duplicate sales can overstate transactions, "
        "units, revenue, and profit."
    )
})

### 2 - Conflicting transaction IDs

In [ ]:
duplicate_id_mask = cleaned_sales[
    "transaction_id"
].duplicated(
    keep=False
)

duplicate_id_count = duplicate_id_mask.sum()

print(
    "Records with duplicate transaction IDs:",
    duplicate_id_count
)

In [ ]:
duplicate_transaction_rows = (
    cleaned_sales[
        duplicate_id_mask
    ]
    .sort_values("transaction_id")
)

duplicate_transaction_rows.head(20)

In [ ]:
duplicate_review = cleaned_sales[
    duplicate_id_mask
].copy()

In [ ]:
cleaned_sales = cleaned_sales[
    ~duplicate_id_mask
].copy()

In [ ]:
cleaning_audit.append({
    "issue": "Conflicting duplicate transaction IDs",
    "action": "Quarantined for review",
    "records_affected": int(duplicate_id_count),
    "business_reason": (
        "Conflicting records cannot be safely resolved "
        "without source-system evidence."
    )
})

### 3 - Clean the unit price data type

In [ ]:
cleaned_sales["unit_price_raw"] = (
    cleaned_sales["unit_price"]
)

In [ ]:
cleaned_sales["unit_price"] = (
    cleaned_sales["unit_price"]
    .astype("string")
    .str.replace(
        r"^\s*SAR\s*",
        "",
        regex=True
    )
    .str.replace(
        ",",
        "",
        regex=False
    )
)

In [ ]:
cleaned_sales["unit_price"] = pd.to_numeric(
    cleaned_sales["unit_price"],
    errors="coerce"
)

In [ ]:
cleaned_sales["unit_price"].dtype

### Identify prices that still can not be converted

In [ ]:
invalid_price_mask = (
    cleaned_sales["unit_price"].isna()
    |
    (cleaned_sales["unit_price"] <= 0)
)

invalid_price_count = invalid_price_mask.sum()

print(
    "Invalid unit prices:",
    invalid_price_count
)

In [ ]:
invalid_price_review = cleaned_sales[
    invalid_price_mask
].copy()

In [ ]:
cleaned_sales = cleaned_sales[
    ~invalid_price_mask
].copy()

In [ ]:
cleaning_audit.append({
    "issue": "Invalid unit price",
    "action": "Quarantined for review",
    "records_affected": int(invalid_price_count),
    "business_reason": (
        "A non-positive or unresolvable selling price "
        "cannot support reliable revenue calculations."
    )
})

### 4  - Quantity

In [ ]:
invalid_quantity_mask = (
    cleaned_sales["quantity"] <= 0
)

invalid_quantity_count = (
    invalid_quantity_mask.sum()
)

print(
    "Invalid quantities:",
    invalid_quantity_count
)

In [ ]:
invalid_quantity_review = cleaned_sales[
    invalid_quantity_mask
].copy()

In [ ]:
cleaned_sales = cleaned_sales[
    ~invalid_quantity_mask
].copy()

In [ ]:
cleaning_audit.append({
    "issue": "Zero or negative quantity",
    "action": "Quarantined for review",
    "records_affected": int(invalid_quantity_count),
    "business_reason": (
        "This fact table represents completed sales. "
        "Returns should be handled separately rather than "
        "silently interpreted as negative sales."
    )
})

### 5 - Missing transaction dates

In [ ]:
cleaned_sales["transaction_date"] = pd.to_datetime(
    cleaned_sales["transaction_date"],
    errors="coerce"
)

In [ ]:
missing_date_mask = (
    cleaned_sales["transaction_date"].isna()
)

missing_date_count = (
    missing_date_mask.sum()
)

print(
    "Missing/invalid dates:",
    missing_date_count
)

In [ ]:
missing_date_review = cleaned_sales[
    missing_date_mask
].copy()

In [ ]:
cleaned_sales = cleaned_sales[
    ~missing_date_mask
].copy()

In [ ]:
cleaning_audit.append({
    "issue": "Missing or invalid transaction date",
    "action": "Quarantined for review",
    "records_affected": int(missing_date_count),
    "business_reason": (
        "A transaction cannot be reliably assigned to a "
        "reporting period without a valid date."
    )
})

### 6 - Future dates

In [ ]:
expected_min_date = pd.Timestamp("2024-01-01")
expected_max_date = pd.Timestamp("2025-12-31")

In [ ]:
future_date_mask = (
    cleaned_sales["transaction_date"]
    > expected_max_date
)

future_date_count = future_date_mask.sum()

print(
    "Future-dated transactions:",
    future_date_count
)

In [ ]:
future_date_review = cleaned_sales[
    future_date_mask
].copy()

In [ ]:
cleaned_sales = cleaned_sales[
    ~future_date_mask
].copy()

In [ ]:
cleaning_audit.append({
    "issue": "Future transaction date",
    "action": "Quarantined for review",
    "records_affected": int(future_date_count),
    "business_reason": (
        "Future dates fall outside the defined source-data "
        "reporting period and can distort time-based KPIs."
    )
})

### 7 - Missing payment methods

In [ ]:
missing_payment_mask = (
    cleaned_sales["payment_method"].isna()
)

missing_payment_count = (
    missing_payment_mask.sum()
)

cleaned_sales["payment_method"] = (
    cleaned_sales["payment_method"]
    .fillna("Unknown")
)

In [ ]:
cleaning_audit.append({
    "issue": "Missing payment method",
    "action": "Replaced with 'Unknown'",
    "records_affected": int(missing_payment_count),
    "business_reason": (
        "The transaction remains valid for sales reporting. "
        "Unknown preserves the sale without inventing a payment method."
    )
})

### 8 - Missing customer IDs

In [ ]:
cleaned_sales["customer_id_raw"] = (
    cleaned_sales["customer_id"]
)

In [ ]:
valid_customer_ids = set(
    customers["customer_id"]
)

missing_customer_mask = (
    cleaned_sales["customer_id"].isna()
)

orphan_customer_mask = (
    cleaned_sales["customer_id"].notna()
    &
    ~cleaned_sales["customer_id"].isin(
        valid_customer_ids
    )
)

In [ ]:
print(
    "Missing customer IDs:",
    missing_customer_mask.sum()
)

print(
    "Orphan customer IDs:",
    orphan_customer_mask.sum()
)

In [ ]:
cleaned_sales.loc[
    missing_customer_mask | orphan_customer_mask,
    "customer_id"
] = "UNKNOWN"

In [ ]:
customer_issue_count = (
    missing_customer_mask
    | orphan_customer_mask
).sum()

cleaning_audit.append({
    "issue": "Missing or orphan customer ID",
    "action": "Mapped to UNKNOWN",
    "records_affected": int(customer_issue_count),
    "business_reason": (
        "The sale remains valid, but customer attribution "
        "is unavailable. UNKNOWN preserves the transaction "
        "without inventing customer information."
    )
})

### 9 - Product and Store oprhan IDs

In [ ]:
cleaned_sales["product_id_raw"] = (
    cleaned_sales["product_id"]
)

cleaned_sales["store_id_raw"] = (
    cleaned_sales["store_id"]
)

In [ ]:
valid_product_ids = set(
    products["product_id"]
)

valid_store_ids = set(
    stores["store_id"]
)

orphan_product_mask = (
    ~cleaned_sales["product_id"]
    .isin(valid_product_ids)
)

orphan_store_mask = (
    ~cleaned_sales["store_id"]
    .isin(valid_store_ids)
)

print(
    "Orphan product IDs:",
    orphan_product_mask.sum()
)

print(
    "Orphan store IDs:",
    orphan_store_mask.sum()
)

In [ ]:
cleaned_sales.loc[
    orphan_product_mask,
    "product_id"
] = "UNKNOWN"

cleaned_sales.loc[
    orphan_store_mask,
    "store_id"
] = "UNKNOWN"

In [ ]:
cleaning_audit.append({
    "issue": "Orphan product ID",
    "action": "Mapped to UNKNOWN",
    "records_affected": int(orphan_product_mask.sum()),
    "business_reason": (
        "The sales event may still be valid, but product "
        "attribution cannot be trusted without a matching master record."
    )
})

cleaning_audit.append({
    "issue": "Orphan store ID",
    "action": "Mapped to UNKNOWN",
    "records_affected": int(orphan_store_mask.sum()),
    "business_reason": (
        "The sales event may still be valid, but store attribution "
        "cannot be trusted without a matching store master record."
    )
})

### 10 - Standardize categories

In [ ]:
cleaning_audit.append({
    "issue": "Orphan product ID",
    "action": "Mapped to UNKNOWN",
    "records_affected": int(orphan_product_mask.sum()),
    "business_reason": (
        "The sales event may still be valid, but product "
        "attribution cannot be trusted without a matching master record."
    )
})

cleaning_audit.append({
    "issue": "Orphan store ID",
    "action": "Mapped to UNKNOWN",
    "records_affected": int(orphan_store_mask.sum()),
    "business_reason": (
        "The sales event may still be valid, but store attribution "
        "cannot be trusted without a matching store master record."
    )
})

In [ ]:
category_standardization_map = {
    "electronics": "Electronics",
    "electronic": "Electronics",
    "home appliances": "Home Appliances",
    "fashion": "Fashion",
    "beauty": "Beauty",
    "sports": "Sports",
    "sport": "Sports",
    "grocery": "Grocery",
    "home & living": "Home & Living"
}

In [ ]:
cleaned_sales["category"] = (
    cleaned_sales["category"]
    .str.lower()
    .map(category_standardization_map)
)

In [ ]:
cleaned_sales["category"].value_counts(
    dropna=False
)

### 11 - Invalid discounts

In [ ]:
invalid_discount_mask = (
    (cleaned_sales["discount_amount"] < 0)
    |
    (
        cleaned_sales["discount_amount"]
        > cleaned_sales["gross_sales"]
    )
)

In [ ]:
invalid_discount_count = (
    invalid_discount_mask.sum()
)

print(
    "Invalid discounts:",
    invalid_discount_count
)

In [ ]:
invalid_discount_review = cleaned_sales[
    invalid_discount_mask
].copy()

cleaned_sales = cleaned_sales[
    ~invalid_discount_mask
].copy()

In [ ]:
cleaning_audit.append({
    "issue": "Invalid discount",
    "action": "Quarantined for review",
    "records_affected": int(invalid_discount_count),
    "business_reason": (
        "The intended discount cannot be reliably inferred. "
        "Capping the value would create an unsupported assumption."
    )
})

### 12 - Recalculate financial fields

In [ ]:
cleaned_sales["gross_sales"] = (
    cleaned_sales["quantity"]
    * cleaned_sales["unit_price"]
).round(2)

In [ ]:
cleaned_sales["net_sales"] = (
    cleaned_sales["gross_sales"]
    - cleaned_sales["discount_amount"]
).round(2)

In [ ]:
cleaned_sales["cost_amount"] = (
    cleaned_sales["quantity"]
    * cleaned_sales["unit_cost"]
).round(2)

In [ ]:
cleaned_sales["profit_amount"] = (
    cleaned_sales["net_sales"]
    - cleaned_sales["cost_amount"]
).round(2)

In [ ]:
cleaned_sales["gross_margin_pct"] = np.where(
    cleaned_sales["net_sales"] != 0,
    (
        cleaned_sales["profit_amount"]
        / cleaned_sales["net_sales"]
        * 100
    ),
    np.nan
).round(2)

In [ ]:
review_frames = [
    duplicate_review,
    invalid_price_review,
    invalid_quantity_review,
    missing_date_review,
    future_date_review,
    invalid_discount_review
]

review_sales = pd.concat(
    review_frames,
    ignore_index=True
).drop_duplicates(
    subset=["source_row_id"]
)

In [ ]:
print(
    "Review records:",
    len(review_sales)
)

In [ ]:
review_sales["review_reason"] = "Manual/source-system review required"

In [ ]:
cleaned_sales.shape

In [ ]:
cleaned_sales.head()

In [ ]:
cleaned_sales.dtypes

In [ ]:
print(
    "Duplicate transaction IDs:",
    cleaned_sales["transaction_id"]
    .duplicated()
    .sum()
)

In [ ]:
print(
    "Exact duplicate rows:",
    cleaned_sales.duplicated().sum()
)

In [ ]:
print(
    "Invalid quantities:",
    (cleaned_sales["quantity"] <= 0).sum()
)

In [ ]:
print(
    "Invalid prices:",
    (
        cleaned_sales["unit_price"] <= 0
    ).sum()
)

In [ ]:
print(
    "Missing dates:",
    cleaned_sales["transaction_date"].isna().sum()
)

print(
    "Future dates:",
    (
        cleaned_sales["transaction_date"]
        > expected_max_date
    ).sum()
)

In [ ]:
gross_check = (
    cleaned_sales["quantity"]
    * cleaned_sales["unit_price"]
).round(2)

print(
    "Gross sales mismatches:",
    ~np.isclose(
        cleaned_sales["gross_sales"],
        gross_check
    ).sum()
)

In [ ]:
gross_mismatch = ~np.isclose(
    cleaned_sales["gross_sales"],
    gross_check,
    rtol=0,
    atol=0.01
)

print(
    "Gross sales mismatches:",
    gross_mismatch.sum()
)

In [ ]:
net_check = (
    cleaned_sales["gross_sales"]
    - cleaned_sales["discount_amount"]
).round(2)

net_mismatch = ~np.isclose(
    cleaned_sales["net_sales"],
    net_check,
    rtol=0,
    atol=0.01
)

print(
    "Net sales mismatches:",
    net_mismatch.sum()
)

In [ ]:
profit_check = (
    cleaned_sales["net_sales"]
    - cleaned_sales["cost_amount"]
).round(2)

profit_mismatch = ~np.isclose(
    cleaned_sales["profit_amount"],
    profit_check,
    rtol=0,
    atol=0.01
)

print(
    "Profit mismatches:",
    profit_mismatch.sum()
)

In [ ]:
invalid_discounts_after = (
    (cleaned_sales["discount_amount"] < 0)
    |
    (
        cleaned_sales["discount_amount"]
        > cleaned_sales["gross_sales"]
    )
)

print(
    "Invalid discounts after cleaning:",
    invalid_discounts_after.sum()
)

In [ ]:
cleaning_audit_df = pd.DataFrame(
    cleaning_audit
)

In [ ]:
cleaning_audit_df

In [ ]:
cleaned_sales_path = (
    CLEANED_DIR / "fact_sales_cleaned.csv"
)

cleaned_sales.to_csv(
    cleaned_sales_path,
    index=False
)

print(
    f"Saved cleaned data: {cleaned_sales_path}"
)

In [ ]:
review_sales_path = (
    CLEANED_DIR / "fact_sales_review.csv"
)

review_sales.to_csv(
    review_sales_path,
    index=False
)

print(
    f"Saved review data: {review_sales_path}"
)

In [ ]:
audit_path = (
    CLEANED_DIR / "cleaning_audit.csv"
)

cleaning_audit_df.to_csv(
    audit_path,
    index=False
)

print(
    f"Saved audit report: {audit_path}"
)

In [ ]:
print("=" * 50)
print("CLEANING SUMMARY")
print("=" * 50)

print(f"Raw records: {len(sales):,}")
print(f"Cleaned records: {len(cleaned_sales):,}")
print(f"Review records: {len(review_sales):,}")

print(
    f"Duplicate transaction IDs: "
    f"{cleaned_sales['transaction_id'].duplicated().sum():,}"
)

print(
    f"Invalid quantities: "
    f"{(cleaned_sales['quantity'] <= 0).sum():,}"
)

print(
    f"Invalid prices: "
    f"{(cleaned_sales['unit_price'] <= 0).sum():,}"
)

print(
    f"Invalid discounts: "
    f"{invalid_discounts_after.sum():,}"
)

print(
    f"Gross sales mismatches: "
    f"{gross_mismatch.sum():,}"
)

print(
    f"Profit mismatches: "
    f"{profit_mismatch.sum():,}"
)